# Plotting global mortality timeseries per 100,000 due to PM<sub>2.5</sub> exposure

Using CESM2, SSP2-4.5 ensemble member 1 as an example

In [ ]:
import os
import glob
import numpy as np
import xarray as xr
import matplotlib
import matplotlib.pyplot as plt
from utils.utils import autosize_figure, get_scenario_config
import config
from utils.utils import require_dir
import pathlib

In [ ]:
def process_all_mortality_per_100k(model, scenario, GBD_version, years, ens_num, n_samples):
    """
    Calculate mortality per 100k for each mortality outcome
    """
    dates = f"{years.start}-{years.stop}"

    # Load population
    POP_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "SSP_pop" / "SSP2")
    pop_file = "ssp2_coarse_grid_annual_2000-2100.nc"
    pop_path = os.path.join(POP_DIR, pop_file)
    population = xr.open_dataarray(pop_path)

    global_pop = population.sum(dim=("lat", "lon"))

    # === Load data ===
    DIR = require_dir(pathlib.Path(config.WORK_ROOT) / model / "mortality" / "pm25")
    in_files = f"Global_mortality_{GBD_version}_*_{n_samples}samples_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
    in_path = os.path.join(DIR, in_files)
    files = sorted(glob.glob(in_path))

    # Open, calculate per 100k and combine all mortality outcomes
    datasets_per100k = [(xr.open_dataarray(f)/global_pop)*100000 for f in files]

    # Align (in case of slight coordinate mismatches)
    da_per100k = xr.align(*datasets_per100k, join="exact")

    return da_per100k

In [ ]:
def plotting_data(da, q1, q2):
    mean = da.mean("samples")
    p1 = da.quantile(q1, "samples")
    p2 = da.quantile(q2, "samples")
    return mean, p1, p2

In [ ]:
def plot_stacked_global_mortality(da, q1, q2, years, model, scenario, GBD_version, ens_num, labels):
    dates = f"{years.start}-{years.stop}"

    # Create magma colours for each category
    cmap = matplotlib.colormaps["magma"]
    colors = [cmap(i) for i in np.linspace(0.25, 0.95, len(da))]

    # Extract mean and quantiles for each health variable
    means = []
    p1s = []
    p2s = []

    for item in da:
        da_mean, da_p1, da_p2 = plotting_data(item, q1, q2)
        means.append(da_mean)
        p1s.append(da_p1)
        p2s.append(da_p2)

    means = np.array(means)
    p1s = np.array(p1s)
    p2s = np.array(p2s)

    # Build the cumulative stacking
    stack_means = np.cumsum(means, axis=0)
    stack_p1s = np.cumsum(p1s, axis=0)
    stack_p2s = np.cumsum(p2s, axis=0)

    years = da_mean["year"].values

    plt.figure(figsize=autosize_figure(1, 1, scale_factor=1.2))
    plt.rcParams.update({'font.size': 16})

    for i in range(len(means)):
        # Lower boundary
        lower_mean = stack_means[i-1] if i > 0 else 0
        upper_mean = stack_means[i]

        # Uncertainty bounds
        p1 = stack_p1s[i]
        p2 = stack_p2s[i]

        # Uncertainty shading
        plt.fill_between(
            years,
            p1,
            p2,
            color="grey",
            alpha=0.25
        )

        # Stacked band
        plt.fill_between(
            years,
            lower_mean,
            upper_mean,
            color=colors[i],
            alpha=0.7,
            label=labels[i],
        )

        plt.plot(years, upper_mean, color="grey", lw=1)

    plt.ylim(bottom=0)
    plt.ticklabel_format(style="plain", axis="y")
    plt.grid(True, alpha=0.2)

    plt.ylabel("Global mortality per 100,000")
    plt.title(f"{model} {scenario} ensemble {ens_num}",)

    plt.legend(
        frameon=True,
        facecolor="white",
        edgecolor="none",
        framealpha=0.8,
        loc='upper center',           # base location
        bbox_to_anchor=(0.5, -0.15),  # position below the plot
        ncol=3                        # 3 items per row → 2 rows for 6 items
    )

    ax = plt.gca()
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.tick_params(axis="x", length=0)
    ax.tick_params(axis="y", length=0)

    plt.tight_layout()

    out_file = f"Global_pm25_mortality_stacked_per100k_{GBD_version}_{model}_{scenario}_{dates}.png"
    out_path = os.path.join(SAVE_DIR, out_file)
    plt.savefig(out_path, dpi=300)

    return

In [ ]:
# === Path Config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
SAVE_DIR = require_dir(pathlib.Path(config.PLOTTING_ROOT) / "example_workflow")

model = "CESM2"
scenario = "SSP245"
GBD_version = "GBD23"
ensemble_number = 1

configs = get_scenario_config(model, scenario)
years = configs["years"]

n_samples = 300

# Quantiles for plotting uncertainty
q1 = 0.25
q2 = 0.75
labels = ["COPD", "Dementia", "Type II Diabetes", "Ischemic Heart Disease",
          "Lower Respiratory Infections", "Lung Cancer",
          "Stroke"]

da_100k = process_all_mortality_per_100k(model, scenario, GBD_version, years, ensemble_number, n_samples,)

plot_stacked_global_mortality(da_100k, q1, q2, years, model, scenario, GBD_version, ensemble_number, labels)